# Sentiment Analysis - Optimized Model Pipeline

This notebook performs EDA, data preprocessing, improved text cleaning with preserved negation words, sublinear TF-IDF vectorization with n-grams (1-3), model training (LinearSVC & Logistic Regression), evaluation, and model serialization.

### Step 1: Import Necessary Libraries & Load Data

In [20]:
import pandas as pd
import numpy as np
import re
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Download NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

# Load dataset from Data directory
file_path = '../Data/Womens Clothing E-Commerce Reviews.csv'
df = pd.read_csv(file_path)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


### Step 2: Keep Relevant Columns & Display First 5 Rows

In [21]:
# Select only 'Review Text' and 'Rating' columns, renamed to 'review' and 'rating'
df = df[['Review Text', 'Rating']].rename(columns={'Review Text': 'review', 'Rating': 'rating'})
df.head(5)

,review,rating
0,Absolutely wonderful - silky and sexy and comf...,4
1,Love this dress! it's sooo pretty. i happene...,5
2,I had such high hopes for this dress and reall...,3
3,"I love, love, love this jumpsuit. it's fun, fl...",5
4,This shirt is very flattering to all due to th...,5


### Step 3: DataFrame Shape & Unique Target Values

In [22]:
# Check shape of the dataframe
print("Shape of DataFrame:", df.shape)

# Unique values in rating
print("Unique values in rating:", df['rating'].unique())

Shape of DataFrame: (23486, 2)
Unique values in rating: [4 5 3 2 1]


### Step 4: Handle Missing/Null Values & Check Updated Shape

In [23]:
# Drop null / missing values
df = df.dropna().reset_index(drop=True)

# Check updated shape
print("Updated Shape after dropping missing values:", df.shape)

Updated Shape after dropping missing values: (22641, 2)


### Step 5: Convert Ratings to Sentiment & Retain Only Review and Sentiment Columns

In [24]:
# Mapping function for sentiment
def map_sentiment(rating):
    if rating in [1, 2]:
        return 'negative'
    elif rating == 3:
        return 'neutral'
    elif rating in [4, 5]:
        return 'positive'

df['sentiment'] = df['rating'].apply(map_sentiment)

# Drop rating column, keeping only review and sentiment columns
df = df[['review', 'sentiment']]
df.head()

,review,sentiment
0,Absolutely wonderful - silky and sexy and comf...,positive
1,Love this dress! it's sooo pretty. i happene...,positive
2,I had such high hopes for this dress and reall...,neutral
3,"I love, love, love this jumpsuit. it's fun, fl...",positive
4,This shirt is very flattering to all due to th...,positive


### Step 6: Text Cleaning (Preserving Negation Words & Removing URLs/Emojis)

In [25]:
# Contractions dictionary mapping
contractions_dict = {
    "can't": "cannot", "won't": "will not", "n't": " not", "'re": " are",
    "'s": " is", "'d": " would", "'ll": " will", "'t": " not",
    "'ve": " have", "'m": " am"
}

def expand_contractions(text, c_dict=contractions_dict):
    pattern = re.compile(r'\b(' + '|'.join(c_dict.keys()) + r')\b')
    return pattern.sub(lambda m: c_dict[m.group(0)], text)

def clean_text(text):
    # 1. Lowercasing
    text = text.lower()
    # 2. Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # 3. Handle contractions
    text = expand_contractions(text)
    # 4. Remove emojis & non-ASCII characters
    text = text.encode('ascii', 'ignore').decode('ascii')
    # 5. Remove special characters and numbers (retain letters and spaces)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # 6. Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply text cleaning
df['cleaned_review'] = df['review'].apply(clean_text)
df[['review', 'cleaned_review']].head()

,review,cleaned_review
0,Absolutely wonderful - silky and sexy and comf...,absolutely wonderful silky and sexy and comfor...
1,Love this dress! it's sooo pretty. i happene...,love this dress it is sooo pretty i happened t...
2,I had such high hopes for this dress and reall...,i had such high hopes for this dress and reall...
3,"I love, love, love this jumpsuit. it's fun, fl...",i love love love this jumpsuit it is fun flirt...
4,This shirt is very flattering to all due to th...,this shirt is very flattering to all due to th...


### Step 7: Lemmatization with Preserved Negations

In [26]:
lemmatizer = WordNetLemmatizer()

# Preserve crucial negation words in stopwords
negation_words = {'no', 'not', 'nor', 'neither', 'never', 'cannot', 'couldnot', 'wouldnot', 'shouldnot', 'isnot', 'cannot'}
stop_words = set(stopwords.words('english')) - negation_words

def lemmatize_text(text):
    words = text.split()
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(lemmatized_words)

df['lemmatized_review'] = df['cleaned_review'].apply(lemmatize_text)
df[['cleaned_review', 'lemmatized_review']].head()

,cleaned_review,lemmatized_review
0,absolutely wonderful silky and sexy and comfor...,absolutely wonderful silky sexy comfortable
1,love this dress it is sooo pretty i happened t...,love dress sooo pretty happened find store gla...
2,i had such high hopes for this dress and reall...,high hope dress really wanted work initially o...
3,i love love love this jumpsuit it is fun flirt...,love love love jumpsuit fun flirty fabulous ev...
4,this shirt is very flattering to all due to th...,shirt flattering due adjustable front tie perf...


### Step 8: Tokenization

In [27]:
# Tokenize text into words
df['tokens'] = df['lemmatized_review'].apply(lambda x: word_tokenize(x))
df[['lemmatized_review', 'tokens']].head()

,lemmatized_review,tokens
0,absolutely wonderful silky sexy comfortable,"[absolutely, wonderful, silky, sexy, comfortable]"
1,love dress sooo pretty happened find store gla...,"[love, dress, sooo, pretty, happened, find, st..."
2,high hope dress really wanted work initially o...,"[high, hope, dress, really, wanted, work, init..."
3,love love love jumpsuit fun flirty fabulous ev...,"[love, love, love, jumpsuit, fun, flirty, fabu..."
4,shirt flattering due adjustable front tie perf...,"[shirt, flattering, due, adjustable, front, ti..."


### Step 9: Train-Test Split & Sublinear TF-IDF Vectorization (N-Grams 1 to 3)

In [28]:
# Target variable and features
X = df['lemmatized_review']
y = df['sentiment']

# Perform Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Expanded Sublinear TF-IDF Vectorizer (unigrams, bigrams, trigrams up to 10,000 features)
tfidf_vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 3), sublinear_tf=True)

# Fit and transform training data, transform test data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("X_train_tfidf shape:", X_train_tfidf.shape)
print("X_test_tfidf shape:", X_test_tfidf.shape)

X_train_tfidf shape: (18112, 10000)
X_test_tfidf shape: (4529, 10000)


### Step 10: Model Training & Comparison (Optimized SVM vs Logistic Regression)

In [29]:
# 1. LinearSVC with tuned C parameter
svm_model = LinearSVC(C=0.5, random_state=42)
svm_model.fit(X_train_tfidf, y_train)

# 2. Logistic Regression model
lr_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)

print("Models successfully trained.")

Models successfully trained.


### Step 11: 3-Class Sentiment Model Evaluation

In [31]:
y_pred_svm = svm_model.predict(X_test_tfidf)
y_pred_lr = lr_model.predict(X_test_tfidf)

print("=== 3-Class LinearSVC Results ===")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred_svm):.4f}\n")
print("Classification Report:\n", classification_report(y_test, y_pred_svm))

print("\n=== 3-Class Logistic Regression Results ===")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred_lr):.4f}\n")
print("Classification Report:\n", classification_report(y_test, y_pred_lr))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))

=== 3-Class LinearSVC Results ===
Accuracy Score: 0.8260

Classification Report:
               precision    recall  f1-score   support

    negative       0.59      0.47      0.53       474
     neutral       0.45      0.25      0.32       565
    positive       0.88      0.97      0.92      3490

    accuracy                           0.83      4529
   macro avg       0.64      0.56      0.59      4529
weighted avg       0.80      0.83      0.81      4529


=== 3-Class Logistic Regression Results ===
Accuracy Score: 0.8245

Classification Report:
               precision    recall  f1-score   support

    negative       0.62      0.42      0.50       474
     neutral       0.46      0.24      0.31       565
    positive       0.87      0.97      0.92      3490

    accuracy                           0.82      4529
   macro avg       0.65      0.55      0.58      4529
weighted avg       0.79      0.82      0.80      4529

Confusion Matrix:
 [[ 201   98  175]
 [  93  134  338]
 [  30  

### Step 12 (Optional Experiment): Binary Classification Performance (Positive vs Negative)

In [32]:
# Filter out 'neutral' class to test binary sentiment classification accuracy
df_binary = df[df['sentiment'] != 'neutral'].copy()

X_bin = df_binary['lemmatized_review']
y_bin = df_binary['sentiment']

X_tr_bin, X_te_bin, y_tr_bin, y_te_bin = train_test_split(X_bin, y_bin, test_size=0.2, random_state=42, stratify=y_bin)

tfidf_binary = TfidfVectorizer(max_features=10000, ngram_range=(1, 3), sublinear_tf=True)
X_tr_bin_tfidf = tfidf_binary.fit_transform(X_tr_bin)
X_te_bin_tfidf = tfidf_binary.transform(X_te_bin)

svm_binary = LinearSVC(C=0.5, random_state=42)
svm_binary.fit(X_tr_bin_tfidf, y_tr_bin)
y_pred_binary = svm_binary.predict(X_te_bin_tfidf)

print("=== Binary Classification (Positive vs Negative) Results ===")
print(f"Binary Accuracy Score: {accuracy_score(y_te_bin, y_pred_binary):.4f}\n")
print("Classification Report:\n", classification_report(y_te_bin, y_pred_binary))

=== Binary Classification (Positive vs Negative) Results ===
Binary Accuracy Score: 0.9395

Classification Report:
               precision    recall  f1-score   support

    negative       0.83      0.62      0.71       474
    positive       0.95      0.98      0.97      3490

    accuracy                           0.94      3964
   macro avg       0.89      0.80      0.84      3964
weighted avg       0.94      0.94      0.94      3964



### Step 13: Save Best Model & Vectorizer Artifacts (`.pkl` files)

In [33]:
# Save tuned model and vectorizer
joblib.dump(svm_model, 'sentiment_model.pkl')
joblib.dump(tfidf_vectorizer, 'vectorizer.pkl')

print("Saved optimized 'sentiment_model.pkl' and 'vectorizer.pkl' successfully!")

Saved optimized 'sentiment_model.pkl' and 'vectorizer.pkl' successfully!
